# Análise do PBIA — Plano Brasileiro de Inteligência Artificial

Este notebook integra uma análise acadêmica de rigor, desenvolvida no âmbito de uma pesquisa em Relações Internacionais dedicada ao estudo comparado das estratégias nacionais de inteligência artificial. Seu objeto específico é o Plano Brasileiro de Inteligência Artificial (PBIA), documento que consolida a estratégia do governo federal brasileiro para o fomento, a regulação e o uso da inteligência artificial no país.

A análise aqui conduzida busca compreender, de modo simultaneamente **quantitativo e qualitativo**, a linguagem empregada pelo documento — os termos, categorias, ênfases retóricas, valores e prioridades estratégicas que estruturam o texto —, de modo a identificar os marcos conceituais e os campos semânticos por meio dos quais o Brasil formula sua política de inteligência artificial.

A partir dessa leitura, pretende-se **posicionar o documento internacionalmente**, comparando a linguagem e as escolhas discursivas de o Brasil com o vocabulário e as ênfases adotados pelos demais países e blocos contemplados neste projeto (Brasil, China, Estados Unidos, Europa e Índia), de modo a mapear convergências, divergências e posicionamentos estratégicos distintivos no debate internacional sobre desenvolvimento e governança de inteligência artificial.

As análises e visualizações produzidas neste notebook seguem as diretrizes metodológicas da **Skill02** (Análise e Visualização Gráfica de Documentos): baseiam-se exclusivamente no campo `texto_completo` do JSON de extração correspondente a este documento, produzido na etapa anterior (Skill01), adotam rigor acadêmico integral na leitura do texto-fonte, evitam generalizações, simplificações e inferências não fundamentadas no texto original, e cada visualização construída é acompanhada de sua respectiva descrição, leitura, interpretação e eventuais limitações metodológicas.


## Análise de Vocabulário — Skill02 + Skill 02_Análise_Vocab_A

Esta seção aplica o **Passo 02** (Skill02 – Análise e Visualização Gráfica de Documentos) em conjunto com o seu complemento especializado, a **Skill 02_Análise_Vocab_A** (Análise de Vocabulário e Termos), a pedido do usuário, para levantar os principais termos e o vocabulário utilizados pelo PBIA.

Do JSON `pbia.json`, são utilizados **exclusivamente** os campos `titulo`, `pais_ou_bloco` e `texto_completo`, conforme o Protocolo de uso do JSON da Skill02. Os campos `elementos_descartados`, `data_extracao`, `data_publicacao` e `fonte` não entram nesta análise — em especial, `elementos_descartados` não tem qualquer valor analítico aqui, servindo apenas de auditoria da Skill01.

A metodologia completa (remoção de stopwords, tratamento de siglas e expressões compostas, agrupamento de variantes morfológicas e critério de corte) está documentada de forma auditável e cumulativa no registro persistente [`pbia_vocab_registro.md`](./pbia_vocab_registro.md), nesta mesma pasta, conforme exigido pelo item 6 da Skill 02_Análise_Vocab_A. Esse registro deve ser consultado — e atualizado — antes de qualquer nova visualização de vocabulário sobre este mesmo documento.

In [ ]:
import json
import re
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

with open("pbia.json", encoding="utf-8") as f:
    pbia = json.load(f)

# Protocolo de uso do JSON (Skill02): apenas estes três campos são utilizados nesta análise
titulo = pbia["titulo"]
pais_ou_bloco = pbia["pais_ou_bloco"]
texto_completo = pbia["texto_completo"]

print(f"Documento: {titulo}")
print(f"País/bloco: {pais_ou_bloco}")
print(f"Tamanho de texto_completo: {len(texto_completo):,} caracteres".replace(",", "."))

### 1. Identificação do idioma e etapas metodológicas (Skill 02_Análise_Vocab_A)

`texto_completo` está integralmente em **inglês** (a publicação oficial do MCTI/CGEE aqui analisada foi extraída em sua versão em língua inglesa — ver Skill01). As stopwords e regras de normalização abaixo são, portanto, do inglês.

O bloco de código a seguir implementa, nesta ordem:
1. **Expressões compostas e siglas equivalentes** — tratadas como unidade única *antes* da tokenização em palavras isoladas, para não fragmentar termos técnicos (ex.: "Artificial Intelligence"/"AI" nunca é contado como as palavras soltas "artificial" + "intelligence").
2. **Tokenização** do texto em palavras.
3. **Remoção** de stopwords do inglês, de marcadores de lista/resíduos não semânticos e do termo de baixo valor analítico "expected" (ver justificativas no registro persistente).
4. **Normalização morfológica** (singular/plural do mesmo lema), agrupando variantes sob um rótulo representativo comum, sem descartar nenhuma ocorrência do cômputo.

In [ ]:
# ============ 1) EXPRESSÕES COMPOSTAS E SIGLAS (mesmo referente) ============
# Ordem: da mais específica/longa para a mais genérica/curta, para evitar
# que um padrão curto "consuma" parte de um padrão mais longo antes da hora.
COMPOUND_TERMS = [
    (r"\bResearch and Development\b|\bR&D\b", "ZZCMPRESEARCHDEV", "Research & Development (R&D)"),
    (r"\bUnified Health System\b|\bSUS\b", "ZZCMPSUS", "Unified Health System (SUS)"),
    (r"\bNational Data Infrastructure\b|\bIND\b", "ZZCMPIND", "National Data Infrastructure (IND)"),
    (r"\bLarge Language Models?\b|\bLLMs?\b", "ZZCMPLLM", "Large Language Models (LLM)"),
    (r"\bSustainable Development Goals?\b|\bSDGs?\b", "ZZCMPSDG", "Sustainable Development Goals (SDGs)"),
    (r"\bArtificial Intelligence\b|\bAI\b", "ZZCMPAI", "Artificial Intelligence (AI)"),
    (r"\bData Centers?\b", "ZZCMPDATACENTER", "Data Center(s)"),
    (r"\bData Infrastructure\b", "ZZCMPDATAINFRA", "Data Infrastructure"),
    (r"\bPublic Sector\b", "ZZCMPPUBSECTOR", "Public Sector"),
    (r"\bPublic Services?\b", "ZZCMPPUBSERVICE", "Public Service(s)"),
    (r"\bPrivate Sector\b", "ZZCMPPRIVSECTOR", "Private Sector"),
    (r"\bValue Chains?\b", "ZZCMPVALUECHAIN", "Value Chain"),
    (r"\bMachine Learning\b", "ZZCMPMACHLEARN", "Machine Learning"),
    (r"\bFederal Government\b", "ZZCMPFEDGOV", "Federal Government"),
    (r"\bDigital Government\b", "ZZCMPDIGGOV", "Digital Government"),
    (r"\bPublic Administration\b", "ZZCMPPUBADMIN", "Public Administration"),
    (r"\bClean Energy\b", "ZZCMPCLEANENERGY", "Clean Energy"),
    (r"\bEnergy Matrix\b", "ZZCMPENERGYMATRIX", "Energy Matrix"),
]

work_text = texto_completo
placeholder_map = {}
for pattern, placeholder, label in COMPOUND_TERMS:
    placeholder_map[placeholder.lower()] = label
    work_text = re.sub(pattern, f" {placeholder} ", work_text, flags=re.IGNORECASE)

# ============ 2) TOKENIZAÇÃO ============
raw_tokens = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ][A-Za-zÀ-ÖØ-öø-ÿ'\-]*", work_text)
tokens = [t.lower() for t in raw_tokens]

# ============ 3) REMOÇÃO (exclusão) ============
ARTICLES = {"a", "an", "the"}
PREPOSITIONS = {"of","in","to","for","with","on","by","as","at","from","into","about","through",
    "during","before","after","above","below","between","under","over","without","within","among",
    "throughout","towards","toward","upon","across","per","via","despite","unlike","regarding","off",
    "out","up","down","besides"}
CONJUNCTIONS = {"and","or","but","nor","so","yet","if","because","while","although","though",
    "whether","since","unless","until","than"}
PRONOUNS_DETERMINERS = {"it","its","this","that","these","those","we","our","ours","they","their",
    "theirs","which","who","whom","whose","i","you","he","she","him","her","us","them","his","hers",
    "itself","themselves","ourselves","yourself","yourselves","himself","herself","one","ones",
    "such","other","others","any","some","each","all","both","either","neither","no","none","own",
    "same"}
AUX_MODAL_VERBS = {"is","are","was","were","be","been","being","am","has","have","had","do","does",
    "did","will","would","shall","should","can","could","may","might","must","ought"}
GENERIC_CONNECTORS = {"more","most","much","many","few","several","various","not","also","only",
    "just","still","even","well","thus","therefore","however","moreover","furthermore","given",
    "whereas"}
STOPWORDS = (ARTICLES | PREPOSITIONS | CONJUNCTIONS | PRONOUNS_DETERMINERS
             | AUX_MODAL_VERBS | GENERIC_CONNECTORS)

# marcadores de lista/seção (A)/B)/C)/D), I)/II)/III)/IV)/V)) — "a" e "i" já cobertos acima
LIST_MARKERS = {"b", "c", "d", "ii", "iii", "iv", "v"}

# termo de template de baixo valor analítico (ver justificativa no registro persistente)
TEMPLATE_TERMS = {"expected"}

# ============ 4) NORMALIZAÇÃO (agrupamento de variantes, sem exclusão) ============
PLURAL_MERGE = {
    "actions": "action(s)", "action": "action(s)",
    "challenges": "challenge(s)", "challenge": "challenge(s)",
    "impacts": "impact(s)", "impact": "impact(s)",
    "solutions": "solution(s)", "solution": "solution(s)",
    "resources": "resource(s)", "resource": "resource(s)",
    "investments": "investment(s)", "investment": "investment(s)",
    "models": "model(s)", "model": "model(s)",
    "benefits": "benefit(s)", "benefit": "benefit(s)",
    "professionals": "professional(s)", "professional": "professional(s)",
    "networks": "network(s)", "network": "network(s)",
    "initiatives": "initiative(s)", "initiative": "initiative(s)",
    "institutions": "institution(s)", "institution": "institution(s)",
    "researchers": "researcher(s)", "researcher": "researcher(s)",
    "capacities": "capacity/capacities", "capacity": "capacity/capacities",
    "companies": "company/companies", "company": "company/companies",
    "citizens": "citizen(s)", "citizen": "citizen(s)",
    "agencies": "agency/agencies", "agency": "agency/agencies",
    "axes": "axis/axes", "axis": "axis/axes",
    "centers": "center(s)", "center": "center(s)",
    "innovations": "innovation(s)", "innovation": "innovation(s)",
    "qualifications": "qualification(s)", "qualification": "qualification(s)",
    "systems": "system(s)", "system": "system(s)",
    "technologies": "technology/technologies", "technology": "technology/technologies",
    "sectors": "sector(s)", "sector": "sector(s)",
    "processes": "process(es)", "process": "process(es)",
    "services": "service(s)", "service": "service(s)",
    "governments": "government(s)", "government": "government(s)",
    "policies": "policy/policies", "policy": "policy/policies",
    "rights": "right(s)", "right": "right(s)",
    "risks": "risk(s)", "risk": "risk(s)",
    "servants": "servant(s)", "servant": "servant(s)",
    "brazil's": "brazil",
}
# "country" (singular, = Brasil no estilo do texto) e "countries" (plural, = outras nações)
# foram deliberadamente NÃO agrupados: designam referentes distintos (ver registro persistente).

freq = Counter()
for t in tokens:
    if t in placeholder_map:
        freq[placeholder_map[t]] += 1
        continue
    if len(t) == 1:
        continue
    if t in LIST_MARKERS:
        continue
    if t in STOPWORDS:
        continue
    if t in TEMPLATE_TERMS:
        continue
    freq[PLURAL_MERGE.get(t, t)] += 1

print(f"Tokens de conteúdo após o pipeline completo: {sum(freq.values()):,}".replace(",", "."))
print(f"Termos únicos após o pipeline completo: {len(freq):,}".replace(",", "."))

In [ ]:
TOP_N = 25  # critério de corte declarado (Skill 02_Análise_Vocab_A, item 4)

top25 = freq.most_common(TOP_N)
df_top25 = pd.DataFrame(top25, columns=["termo", "frequência"])
df_top25.index = range(1, len(df_top25) + 1)
df_top25.index.name = "ranking"
df_top25

In [ ]:
terms = [t for t, n in top25][::-1]
counts = [n for t, n in top25][::-1]

# Paleta validada (skill "dataviz" deste ambiente): slot 1 azul / slot 2 laranja,
# par categórico com CVD ΔE 9.1 (claro) / 8.4 (escuro) — acima do alvo de 8.
COLOR_MAIN = "#2a78d6"
COLOR_HIGHLIGHT = "#eb6834"
colors = [COLOR_HIGHLIGHT if t == "Artificial Intelligence (AI)" else COLOR_MAIN for t in terms]

fig, ax = plt.subplots(figsize=(10, 11.5), dpi=150)
bars = ax.barh(terms, counts, color=colors, height=0.68, zorder=3)

for rect, val in zip(bars, counts):
    ax.text(rect.get_width() + max(counts) * 0.01, rect.get_y() + rect.get_height() / 2,
             f"{val}", va="center", ha="left", fontsize=9, color="#33322f")

ax.set_xlabel("Frequência (nº de ocorrências em texto_completo)", fontsize=10, color="#33322f")

fig.suptitle("PBIA – Termos mais frequentes do vocabulário (Top 25)",
             fontsize=13.5, fontweight="bold", color="#0b0b0b", x=0.02, y=0.99, ha="left")
fig.text(0.02, 0.965,
         "Após remoção de stopwords, normalização morfológica e de siglas/expressões compostas "
         "— Skill02 + Skill 02_Análise_Vocab_A",
         fontsize=8.5, color="#6b6a66", ha="left")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_color("#c9c8c2")
ax.tick_params(axis="y", length=0, labelsize=9.5)
ax.tick_params(axis="x", labelsize=9)
ax.xaxis.grid(True, color="#e6e5e0", zorder=0)
ax.set_axisbelow(True)
ax.set_xlim(0, max(counts) * 1.12)

legend_elems = [
    Patch(facecolor=COLOR_HIGHLIGHT, label="Artificial Intelligence (AI) — termo dominante e esperado"),
    Patch(facecolor=COLOR_MAIN, label="Demais termos (posições 2–25)"),
]
ax.legend(handles=legend_elems, loc="lower right", frameon=False, fontsize=8.5)

plt.tight_layout(rect=[0, 0, 1, 0.955])
plt.show()

### 2. Análise final (Skill02, item 4 · Skill 02_Análise_Vocab_A, item 8)

**O que foi construído.** Um gráfico de barras horizontais com os 25 termos de maior frequência absoluta no `texto_completo` do PBIA, após a aplicação integral do pipeline metodológico descrito acima e detalhado, de forma auditável, em [`pbia_vocab_registro.md`](./pbia_vocab_registro.md). Optou-se por um gráfico de barras — e não por uma nuvem de palavras — porque a barra codifica magnitude de forma precisa e comparável (comprimento sobre um eixo numérico com rótulos diretos de valor), ao passo que nuvens de palavras codificam frequência por tamanho de fonte de maneira perceptualmente imprecisa, o que fere o critério de rigor e precisão exigido pela Skill02.

**Como ler o gráfico.** Cada barra corresponde a um termo (ou a um grupo de variantes equivalentes — sigla+forma por extenso, singular+plural, ou expressão composta) e seu comprimento é o número de ocorrências desse termo em todo o corpo do texto extraído. A barra laranja isola "Artificial Intelligence (AI)"; as demais 24 barras, em azul, compõem o restante do perfil vocabular do documento.

**Interpretação à luz do conteúdo do documento.** "Artificial Intelligence (AI)" domina com folga (627 ocorrências) — resultado esperado e, em certo sentido, tautológico para um plano nacional inteiramente dedicado a IA; por isso ele foi destacado visualmente à parte, para que a leitura analítica se concentre no que vem depois dele. Entre as posições 2–25, o vocabulário do PBIA revela três camadas: (i) **termos de ação/estrutura de política** — "development", "action(s)", "impact(s)", "challenge(s)", "solution(s)", "increase", "support", "program" — que refletem o caráter normativo e orientado a resultados do documento (nota-se, porém, que parte de "action(s)", "impact(s)" e "challenge(s)" é inflada pelos rótulos fixos "Action N:"/"Challenge:"/"Expected impact(s):", repetidos em cada uma das 81 entradas dos Anexos 1-2 — ver quantificação exata no registro persistente); (ii) **termos de soberania/identidade nacional** — "brazil", "national", "brazilian" somam, juntos, mais ocorrências (318) do que qualquer termo isolado depois de "AI", o que é coerente com a ênfase do PBIA em soberania tecnológica e "IA para o bem de todos" com identidade brasileira; e (iii) **termos técnico-setoriais** — "technology/technologies", "technological", "data", "innovation(s)", "system(s)", "infrastructure", "research", "education", "health" — que mapeiam os eixos estruturantes e as áreas prioritárias do Plano (infraestrutura, pesquisa, educação, saúde).

**Ajustes possíveis.** (a) Separar, com anotação estrutural mais explícita no próprio texto de origem, as ocorrências de "action(s)"/"impact(s)"/"challenge(s)" que são rótulo de campo das que são prosa corrida, para um gráfico ainda mais fiel à ênfase conceitual (hoje essa distinção só é dada textualmente, não visualmente); (b) considerar um segundo gráfico em escala logarítmica no eixo X, ou a exclusão pontual e explicitamente justificada de "AI" em uma segunda versão, para melhor discriminar visualmente as diferenças entre os termos de posições 2 a 25, cuja amplitude (152 a 39) é bem menor que a amplitude entre "AI" e o segundo colocado; (c) expandir a lista de expressões compostas tratadas como unidade (ex.: "deep learning", "sustainable development", "public health") caso análises futuras sobre este mesmo documento demandem maior granularidade nesses conceitos.

**Limitações e fraquezas metodológicas.** (a) O critério de fusão de expressões compostas depende de adjacência textual exata (bigrama); variantes não adjacentes do mesmo conceito (ex.: "clean and renewable... energy matrix", em vez de "clean energy matrix") não são capturadas e permanecem como palavras soltas — validado no registro persistente com os dois casos residuais de "clean" fora de "Clean Energy". (b) A normalização morfológica (seção 2.3 do registro) cobre apenas os pares singular/plural de maior frequência, identificados manualmente a partir da lista de tokens mais recorrentes; pares de baixíssima frequência fora dessa lista podem permanecer não normalizados, sem, contudo, afetar o Top 25 aqui apresentado. (c) A contagem é de frequência absoluta, não de frequência relativa/ponderada por seção — um termo concentrado em poucas seções longas (ex.: os Anexos, que respondem por parte substancial do texto) pesa da mesma forma que um termo disperso ao longo de todo o documento; isso é adequado para uma análise individual deste documento (não há comparação entre corpora de tamanhos distintos, o que tornaria a normalização relativa obrigatória, conforme a Skill02), mas é uma limitação a ter em mente na leitura.